# Construcción de un analizador ascendente LR(0)

**Objetivo:** aumentar la gramática, construir items, transiciones, tabla ACTION/GOTO y simular `abb`.

Gramática aumentada:

- (0) S′ → S
- (1) S → S b
- (2) S → a

Un *item LR(0)* indica mediante el punto qué parte de una producción ha sido reconocida.

In [1]:
reglas = [("S'", ("S",)), ("S", ("S", "b")), ("S", ("a",))]
no_terminales = {"S'", "S"}
terminales = {'a','b'}

def closure(items):
    resultado = set(items)
    while True:
        nuevo = set(resultado)
        for indice, punto in resultado:
            derecha = reglas[indice][1]
            if punto < len(derecha) and derecha[punto] in no_terminales:
                simbolo = derecha[punto]
                nuevo |= {(i, 0) for i, (lhs, _) in enumerate(reglas) if lhs == simbolo}
        if nuevo == resultado: return frozenset(resultado)
        resultado = nuevo

def goto(items, simbolo):
    movidos = {(i, punto + 1) for i, punto in items
               if punto < len(reglas[i][1]) and reglas[i][1][punto] == simbolo}
    return closure(movidos) if movidos else frozenset()

estados = [closure({(0, 0)})]
transiciones = {}
for i, estado in enumerate(estados):
    for simbolo in ['S', 'a', 'b']:
        destino = goto(estado, simbolo)
        if destino:
            if destino not in estados: estados.append(destino)
            transiciones[(i, simbolo)] = estados.index(destino)

def item_str(i,p):
    lhs, rhs = reglas[i]
    return f"{lhs} → {' '.join(rhs[:p])} · {' '.join(rhs[p:])}"
for i, estado in enumerate(estados):
    print(f'\nI{i}:')
    for item in sorted(estado): print(' ', item_str(*item))
print('\nTransiciones:', transiciones)


I0:
  S' →  · S
  S →  · S b
  S →  · a

I1:
  S' → S · 
  S → S · b

I2:
  S → a · 

I3:
  S → S b · 

Transiciones: {(0, 'S'): 1, (0, 'a'): 2, (1, 'b'): 3}


## Construcción de ACTION/GOTO

SHIFT mueve el símbolo leído a la pila y cambia de estado. REDUCE aplica una producción completa. ACCEPT aparece al reconocer `S′ → S ·` con fin de entrada.

In [2]:
ACTION, GOTO = {}, {}
for (i, simbolo), j in transiciones.items():
    if simbolo in terminales: ACTION[(i, simbolo)] = ('SHIFT', j)
    else: GOTO[(i, simbolo)] = j
for i, estado in enumerate(estados):
    for numero, punto in estado:
        if punto == len(reglas[numero][1]):
            if numero == 0: ACTION[(i, '$')] = ('ACCEPT',)
            else:
                for simbolo in ('a', 'b', '$'):
                    clave = (i, simbolo)
                    candidato = ('REDUCE', numero)
                    if clave in ACTION and ACTION[clave] != candidato:
                        raise ValueError(f'Conflicto LR(0) en {clave}')
                    ACTION[clave] = candidato
print('Estado | a | b | $ | GOTO(S)')
for i in range(len(estados)):
    print(i, *[ACTION.get((i,s),'-') for s in ('a','b','$')], GOTO.get((i,'S'),'-'), sep=' | ')

Estado | a | b | $ | GOTO(S)
0 | ('SHIFT', 2) | - | - | 1
1 | - | ('SHIFT', 3) | ('ACCEPT',) | -
2 | ('REDUCE', 2) | ('REDUCE', 2) | ('REDUCE', 2) | -
3 | ('REDUCE', 1) | ('REDUCE', 1) | ('REDUCE', 1) | -


## Simulación de `abb$`

La pila almacena alternativamente símbolos y estados. La tabla resultante documenta la pila, la entrada pendiente y la acción ejecutada.

In [3]:
entrada = list('abb') + ['$']
pila = [0]
pos = 0
pasos=[]
while True:
    estado = pila[-1]
    accion = ACTION.get((estado, entrada[pos]))
    if accion is None: raise SyntaxError(f'Sin acción para estado {estado} y {entrada[pos]}')
    pasos.append((' '.join(map(str,pila)), ''.join(entrada[pos:]), str(accion)))
    if accion[0] == 'SHIFT':
        pila.extend([entrada[pos], accion[1]])
        pos += 1
    elif accion[0] == 'REDUCE':
        lhs, rhs = reglas[accion[1]]
        del pila[-2*len(rhs):]
        pila.extend([lhs, GOTO[(pila[-1], lhs)]])
    else: break
print('PILA | ENTRADA | ACCIÓN')
for fila in pasos: print(' | '.join(fila))
assert pasos[-1][-1] == "('ACCEPT',)"
print('Conclusión: cadena abb aceptada.')

PILA | ENTRADA | ACCIÓN
0 | abb$ | ('SHIFT', 2)
0 a 2 | bb$ | ('REDUCE', 2)
0 S 1 | bb$ | ('SHIFT', 3)
0 S 1 b 3 | b$ | ('REDUCE', 1)
0 S 1 | b$ | ('SHIFT', 3)
0 S 1 b 3 | $ | ('REDUCE', 1)
0 S 1 | $ | ('ACCEPT',)
Conclusión: cadena abb aceptada.


## Conclusión

La gramática admite `a` seguida de cero o más símbolos `b`. La cadena `abb` pertenece al lenguaje y es aceptada por el análisis ascendente LR(0).